In [1]:
import os
import re
import json
import time
import html
from pathlib import Path
from typing import Optional

import pandas as pd
from bs4 import BeautifulSoup
from openai import OpenAI
from tqdm.auto import tqdm

In [2]:
# ============================================================
# PATH CONFIGURATION
# ============================================================

# CSV containing the 100 companies
CSV_PATH = Path("sp500.csv")

# Folder containing files such as:
# AAPL_10-K_2025-10-31.html
# AMZN_10-K_2026-02-06.html
FILINGS_DIR = Path("10k-filings")

# Output directory
QUESTIONS_DIR = Path("questions")
QUESTIONS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# MODEL CONFIGURATION
# ============================================================

BASE_URL = "http://api.llm.apps.os.dcs.gla.ac.uk/v1"

# Use this outside the University of Glasgow network instead:
# BASE_URL = "http://api.terrier.org/v1"

API_KEY = os.environ["IDA_LLM_API_KEY"]

MODEL_NAME = "gemma-4-26b-a4b"

client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY
)


# ============================================================
# PROCESSING CONFIGURATION
# ============================================================

NUMBER_OF_QUESTIONS = 30

# Maximum amount of filing text sent for each company.
# 60,000 characters is approximately 12,000–18,000 tokens,
# depending on the filing.
MAX_FILING_CHARACTERS = 100000

# Size of text chunks used when selecting relevant filing content.
CHUNK_SIZE = 4000

# Delay between companies to avoid overloading the cluster API.
DELAY_BETWEEN_REQUESTS = 2

# Number of API attempts per company.
MAX_RETRIES = 3

# If False, companies with an existing output file are skipped.
OVERWRITE_EXISTING = False

In [3]:
companies = pd.read_csv(CSV_PATH)

required_columns = [
    "Symbol",
    "Security",
    "GICS Sector",
    "GICS Sub-Industry"
]

missing_columns = [
    column for column in required_columns
    if column not in companies.columns
]

if missing_columns:
    raise ValueError(
        f"The CSV is missing these required columns: {missing_columns}"
    )

# Remove rows without a ticker or company name.
companies = companies.dropna(
    subset=["Symbol", "Security"]
).copy()

companies["Symbol"] = companies["Symbol"].astype(str).str.strip()
companies["Security"] = companies["Security"].astype(str).str.strip()

print(f"Number of companies loaded: {len(companies)}")
display(companies.head())

Number of companies loaded: 100


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,GOOGL,Alphabet Inc. (Class A),Communication Services,Interactive Media & Services,"Mountain View, California",03-04-2006,1652044,1998
1,T,AT&T,Communication Services,Integrated Telecommunication Services,"Dallas, Texas",30-11-1983,732717,1983 (1885)
2,EA,Electronic Arts,Communication Services,Interactive Home Entertainment,"Redwood City, California",22-07-2002,712515,1982
3,META,Meta Platforms,Communication Services,Interactive Media & Services,"Menlo Park, California",23-12-2013,1326801,2004
4,NFLX,Netflix,Communication Services,Movies & Entertainment,"Los Gatos, California",20-12-2010,1065280,1997


In [4]:
def normalize_ticker(ticker: str) -> str:
    """
    Normalise a ticker for matching.

    This helps with cases such as:
    BRK.B, BRK-B and BRK_B
    """
    return re.sub(r"[^A-Z0-9]", "", str(ticker).upper())


def extract_ticker_from_filename(file_path: Path) -> Optional[str]:
    """
    Extract the ticker from a filename following:
    TICKER_10-K_YYYY-MM-DD.html
    """
    match = re.match(
        r"^(.+?)_10-?K_",
        file_path.stem,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(1)

    return None


def build_filing_index(filings_directory: Path) -> dict:
    """
    Build a dictionary mapping normalised tickers to filing paths.

    If multiple filings exist for one company, all matching paths are
    retained and later sorted by their filing date.
    """
    if not filings_directory.exists():
        raise FileNotFoundError(
            f"10-K directory does not exist: {filings_directory.resolve()}"
        )

    filing_index = {}

    accepted_extensions = {
        ".html",
        ".htm",
        ".txt"
    }

    for file_path in filings_directory.rglob("*"):
        if not file_path.is_file():
            continue

        if file_path.suffix.lower() not in accepted_extensions:
            continue

        ticker_from_filename = extract_ticker_from_filename(file_path)

        if ticker_from_filename is None:
            continue

        normalised_ticker = normalize_ticker(ticker_from_filename)

        filing_index.setdefault(normalised_ticker, []).append(file_path)

    return filing_index

In [5]:
def extract_filing_date(file_path: Path) -> str:
    """
    Extract YYYY-MM-DD from a filing filename.

    Returns an empty string when no date can be identified.
    """
    match = re.search(
        r"(\d{4}-\d{2}-\d{2})",
        file_path.stem
    )

    return match.group(1) if match else ""


def find_company_filing(
    ticker: str,
    filing_index: dict
) -> Optional[Path]:
    """
    Find the latest matching 10-K filing for a ticker.
    """
    normalised_ticker = normalize_ticker(ticker)
    matches = filing_index.get(normalised_ticker, [])

    if not matches:
        return None

    # Sorting YYYY-MM-DD strings works chronologically.
    matches = sorted(
        matches,
        key=extract_filing_date,
        reverse=True
    )

    return matches[0]

In [6]:
filing_index = build_filing_index(FILINGS_DIR)

print(f"Ticker entries found in filing directory: {len(filing_index)}")

matched_filings = []
missing_filings = []

for _, row in companies.iterrows():
    ticker = row["Symbol"]
    filing_path = find_company_filing(ticker, filing_index)

    if filing_path:
        matched_filings.append(
            {
                "Ticker": ticker,
                "Company": row["Security"],
                "Filing": filing_path.name
            }
        )
    else:
        missing_filings.append(
            {
                "Ticker": ticker,
                "Company": row["Security"]
            }
        )

print(f"Companies with filings: {len(matched_filings)}")
print(f"Companies without filings: {len(missing_filings)}")

if missing_filings:
    display(pd.DataFrame(missing_filings))

Ticker entries found in filing directory: 100
Companies with filings: 100
Companies without filings: 0


In [7]:
def read_file_with_fallback_encoding(file_path: Path) -> str:
    """
    Read a filing while handling common SEC filing encodings.
    """
    encodings = [
        "utf-8",
        "utf-8-sig",
        "latin-1",
        "cp1252"
    ]

    for encoding in encodings:
        try:
            return file_path.read_text(
                encoding=encoding,
                errors="strict"
            )
        except UnicodeDecodeError:
            continue

    # Final fallback
    return file_path.read_text(
        encoding="utf-8",
        errors="ignore"
    )

In [8]:
def clean_filing_html(file_path: Path) -> str:
    """
    Convert an HTML or text 10-K filing into clean plain text.
    """
    raw_content = read_file_with_fallback_encoding(file_path)

    if file_path.suffix.lower() in {".html", ".htm"}:
        soup = BeautifulSoup(raw_content, "lxml")

        # Remove elements that do not provide useful filing content.
        for element in soup([
            "script",
            "style",
            "noscript",
            "svg",
            "meta",
            "link"
        ]):
            element.decompose()

        text = soup.get_text(separator=" ")
    else:
        text = raw_content

    text = html.unescape(text)

    # Remove non-breaking spaces and excessive whitespace.
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [9]:
IMPORTANT_FILING_TERMS = {
    "risk factors": 8,
    "management discussion": 8,
    "management's discussion": 8,
    "results of operations": 7,
    "business": 4,
    "revenue": 5,
    "net income": 5,
    "operating income": 5,
    "operating margin": 5,
    "gross margin": 5,
    "cash flow": 5,
    "free cash flow": 5,
    "liquidity": 6,
    "capital resources": 6,
    "capital expenditure": 5,
    "debt": 5,
    "borrowings": 4,
    "interest rate": 4,
    "competition": 6,
    "competitors": 5,
    "market risk": 6,
    "foreign currency": 4,
    "supply chain": 5,
    "customer concentration": 6,
    "supplier concentration": 5,
    "regulation": 5,
    "regulatory": 5,
    "litigation": 5,
    "cybersecurity": 5,
    "acquisition": 5,
    "merger": 5,
    "impairment": 4,
    "segment": 4,
    "geographic": 3,
    "research and development": 4,
    "seasonality": 4,
    "outlook": 6,
    "guidance": 6
}

In [10]:
def split_text_into_chunks(
    text: str,
    chunk_size: int = CHUNK_SIZE
) -> list[str]:
    """
    Split filing text into chunks, preferably at sentence boundaries.
    """
    sentences = re.split(
        r"(?<=[.!?])\s+",
        text
    )

    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence_length = len(sentence)

        if (
            current_chunk
            and current_length + sentence_length > chunk_size
        ):
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_length = 0

        current_chunk.append(sentence)
        current_length += sentence_length + 1

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [11]:
def score_filing_chunk(chunk: str) -> float:
    """
    Score a filing chunk according to its relevance for generating
    stock-performance questions.
    """
    lower_chunk = chunk.lower()

    score = 0.0

    for term, weight in IMPORTANT_FILING_TERMS.items():
        occurrences = lower_chunk.count(term)

        # Limit repeated occurrences so a single term does not dominate.
        score += min(occurrences, 4) * weight

    # Give some preference to chunks containing financial values.
    number_count = len(
        re.findall(
            r"\$[\d,.]+|\d+(?:\.\d+)?%",
            chunk
        )
    )

    score += min(number_count, 10) * 0.5

    return score

In [12]:
def select_relevant_filing_text(
    filing_text: str,
    max_characters: int = MAX_FILING_CHARACTERS
) -> str:
    """
    Select filing chunks most relevant to future stock analysis.

    The beginning of the filing is retained for basic context, while
    the remaining space is filled with high-scoring chunks.
    """
    if len(filing_text) <= max_characters:
        return filing_text

    chunks = split_text_into_chunks(filing_text)

    if not chunks:
        return filing_text[:max_characters]

    # Keep initial filing context.
    selected_chunks = []
    selected_indices = set()
    total_characters = 0

    initial_chunk_count = min(3, len(chunks))

    for index in range(initial_chunk_count):
        selected_chunks.append((index, chunks[index]))
        selected_indices.add(index)
        total_characters += len(chunks[index])

    scored_chunks = [
        (score_filing_chunk(chunk), index, chunk)
        for index, chunk in enumerate(chunks)
        if index not in selected_indices
    ]

    scored_chunks.sort(
        key=lambda item: item[0],
        reverse=True
    )

    for score, index, chunk in scored_chunks:
        if total_characters + len(chunk) > max_characters:
            continue

        selected_chunks.append((index, chunk))
        selected_indices.add(index)
        total_characters += len(chunk)

        if total_characters >= max_characters:
            break

    # Put selected sections back into their original filing order.
    selected_chunks.sort(key=lambda item: item[0])

    return "\n\n".join(
        chunk for _, chunk in selected_chunks
    )

In [13]:
def create_question_generation_prompt(
    company_name: str,
    ticker: str,
    sector: str,
    sub_industry: str,
    filing_date: str,
    filing_text: str
) -> str:
    """
    Construct a detailed prompt for generating exactly 30 questions.
    """
    return f"""
You are a financial research question-generation system.

Your task is to create exactly {NUMBER_OF_QUESTIONS} analytical questions
for monitoring and predicting the future stock performance of one publicly
listed company.

COMPANY INFORMATION
Company name: {company_name}
Ticker: {ticker}
GICS sector: {sector}
GICS sub-industry: {sub_industry}
10-K filing date: {filing_date}

PURPOSE
The questions will later be asked repeatedly using current financial,
market, company, industry and macroeconomic information. Changes in their
answers will be studied to determine whether they contain signals about
the company's future stock performance.

The attached 10-K is supplied only as background for generating informed
and company-specific questions. Do not answer the questions. Do not merely
summarise the filing.

INSTRUCTIONS

1. Produce exactly {NUMBER_OF_QUESTIONS} questions.

2. Every item must be written as a complete question.

3. Create questions specifically for {company_name}. Use:
   - the company's name and ticker;
   - its sector and sub-industry;
   - material business characteristics visible in the 10-K;
   - products, services, markets and business segments identified in the filing;
   - revenue drivers and cost drivers;
   - company-specific risks and opportunities;
   - your existing general knowledge of the company where useful.

4. The questions must be useful for evaluating or predicting stock
   performance over future short-, medium- or long-term periods.

5. Prefer questions whose answers could change over time and could therefore
   be monitored repeatedly. Questions should examine measurable developments,
   trends, events, comparisons or changes rather than static facts.

6. Include a balanced collection of questions covering relevant areas such as:
   - revenue growth and demand;
   - pricing power and unit economics;
   - margins, expenses and profitability;
   - cash flow and capital expenditure;
   - balance-sheet strength, debt and liquidity;
   - business segments, products and geographic markets;
   - customer, supplier or partner concentration;
   - competition and market share;
   - management guidance and earnings expectations;
   - valuation and investor expectations;
   - regulatory, legal, geopolitical and cybersecurity risks;
   - technological disruption and innovation;
   - acquisitions, divestitures and capital allocation;
   - sector-specific economic indicators;
   - macroeconomic sensitivity;
   - potential positive and negative catalysts.

7. Sector relevance is essential. Do not force irrelevant metrics onto the
   company. For example, use subscriber metrics only when subscriptions are
   material, same-store sales only for relevant retailers, and production
   volumes only for companies where production is a meaningful driver.

8. Make each question sufficiently precise that it could later be answered
   using company reports, market data, news, analyst information or other
   current sources.

9. Questions should encourage analysis rather than simple fact retrieval.
   Where appropriate, ask about:
   - direction and magnitude of change;
   - comparisons with prior periods;
   - comparisons with management guidance;
   - comparisons with competitors or sector benchmarks;
   - possible effect on revenue, earnings, cash flow, valuation or investor
     sentiment.

10. Avoid:
    - duplicate or near-duplicate questions;
    - generic questions that could be copied unchanged to almost any company;
    - questions asking only for the current stock price;
    - questions asking for a direct stock-price prediction without supporting
      analysis;
    - questions based on unsupported assumptions;
    - questions whose answers are permanently fixed;
    - questions that reveal their answers inside the question;
    - multi-part questions containing several unrelated topics.

11. Do not cite page numbers because HTML filing pagination may be unreliable.

12. Do not provide explanations, headings, categories, introductory text,
    conclusions or answers.

OUTPUT FORMAT

Return only a valid JSON array containing exactly {NUMBER_OF_QUESTIONS}
strings.

Required structure:

[
  "First complete question?",
  "Second complete question?",
  "Third complete question?"
]

Do not wrap the JSON in Markdown code fences.

10-K FILING EXTRACT
-------------------
{filing_text}
-------------------
END OF 10-K FILING EXTRACT
""".strip()

In [14]:
def remove_markdown_code_fence(text: str) -> str:
    """
    Remove optional Markdown JSON fences.
    """
    text = text.strip()

    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    return text.strip()

In [15]:
def extract_json_array(response_text: str) -> list:
    """
    Extract a JSON array from a model response.
    """
    cleaned_text = remove_markdown_code_fence(response_text)

    try:
        parsed = json.loads(cleaned_text)
    except json.JSONDecodeError:
        # Attempt to locate the first complete-looking JSON array.
        start_position = cleaned_text.find("[")
        end_position = cleaned_text.rfind("]")

        if start_position == -1 or end_position == -1:
            raise ValueError(
                "No JSON array was found in the model response."
            )

        possible_json = cleaned_text[
            start_position:end_position + 1
        ]

        parsed = json.loads(possible_json)

    if not isinstance(parsed, list):
        raise ValueError(
            "The model response is not a JSON array."
        )

    return parsed

In [16]:
def validate_questions(
    questions: list,
    expected_count: int = NUMBER_OF_QUESTIONS
) -> list[str]:
    """
    Ensure the response contains exactly the required number of
    non-empty, unique questions.
    """
    if len(questions) != expected_count:
        raise ValueError(
            f"Expected {expected_count} questions, "
            f"but received {len(questions)}."
        )

    cleaned_questions = []

    for question_number, question in enumerate(
        questions,
        start=1
    ):
        if not isinstance(question, str):
            raise ValueError(
                f"Question {question_number} is not a string."
            )

        question = re.sub(
            r"\s+",
            " ",
            question
        ).strip()

        # Remove numbering if the model included it inside the string.
        question = re.sub(
            r"^\s*\d+\s*[\.\)\-:]\s*",
            "",
            question
        )

        if not question:
            raise ValueError(
                f"Question {question_number} is empty."
            )

        if not question.endswith("?"):
            question += "?"

        cleaned_questions.append(question)

    normalised_questions = [
        re.sub(r"[^a-z0-9]+", " ", question.lower()).strip()
        for question in cleaned_questions
    ]

    if len(set(normalised_questions)) != expected_count:
        raise ValueError(
            "The response contains duplicate questions."
        )

    return cleaned_questions

In [17]:
def generate_questions(
    prompt: str,
    max_retries: int = MAX_RETRIES
) -> list[str]:
    """
    Call the university LLM and return a validated list of questions.
    """
    latest_error = None
    current_prompt = prompt

    for attempt in range(1, max_retries + 1):
        try:
            result = client.responses.create(
                model=MODEL_NAME,
                input=current_prompt
            )

            response_text = result.output_text

            if not response_text or not response_text.strip():
                raise ValueError(
                    "The model returned an empty response."
                )

            questions = extract_json_array(response_text)
            questions = validate_questions(questions)

            return questions

        except Exception as error:
            latest_error = error

            print(
                f"Attempt {attempt}/{max_retries} failed: {error}"
            )

            if attempt < max_retries:
                current_prompt = f"""
{prompt}

IMPORTANT CORRECTION:
Your previous response was invalid.

Return only one valid JSON array containing exactly
{NUMBER_OF_QUESTIONS} unique question strings. Do not include headings,
explanations, numbering outside the strings, or Markdown code fences.
""".strip()

                wait_time = attempt * 5
                print(f"Retrying in {wait_time} seconds...")
                time.sleep(wait_time)

    raise RuntimeError(
        f"Question generation failed after {max_retries} attempts. "
        f"Last error: {latest_error}"
    )

In [18]:
def safe_filename(name: str) -> str:
    """
    Convert a company name into a filename that works on Windows
    and Linux.
    """
    name = re.sub(
        r'[<>:"/\\|?*]',
        "_",
        str(name)
    )

    name = re.sub(r"\s+", " ", name).strip()
    name = name.rstrip(". ")

    return name

In [19]:
def save_questions(
    company_name: str,
    questions: list[str],
    output_directory: Path
) -> Path:
    """
    Save questions as a numbered plain-text file.
    """
    output_filename = f"{safe_filename(company_name)}.txt"
    output_path = output_directory / output_filename

    output_text = "\n".join(
        f"{number}. {question}"
        for number, question in enumerate(
            questions,
            start=1
        )
    )

    output_path.write_text(
        output_text,
        encoding="utf-8"
    )

    return output_path

In [20]:
processing_results = []

for row_number, row in tqdm(
    companies.iterrows(),
    total=len(companies),
    desc="Generating company questions"
):
    ticker = str(row["Symbol"]).strip()
    company_name = str(row["Security"]).strip()

    sector = (
        str(row["GICS Sector"]).strip()
        if pd.notna(row["GICS Sector"])
        else "Not provided"
    )

    sub_industry = (
        str(row["GICS Sub-Industry"]).strip()
        if pd.notna(row["GICS Sub-Industry"])
        else "Not provided"
    )

    output_path = (
        QUESTIONS_DIR
        / f"{safe_filename(company_name)}.txt"
    )

    # Resume support: skip companies already processed.
    if output_path.exists() and not OVERWRITE_EXISTING:
        print(
            f"Skipping {ticker} - output already exists: "
            f"{output_path.name}"
        )

        processing_results.append(
            {
                "Ticker": ticker,
                "Company": company_name,
                "Status": "Skipped - output exists",
                "Filing": "",
                "Output": str(output_path),
                "Error": ""
            }
        )

        continue

    filing_path = find_company_filing(
        ticker,
        filing_index
    )

    if filing_path is None:
        error_message = (
            f"No matching filing found for ticker {ticker}"
        )

        print(error_message)

        processing_results.append(
            {
                "Ticker": ticker,
                "Company": company_name,
                "Status": "Failed",
                "Filing": "",
                "Output": "",
                "Error": error_message
            }
        )

        continue

    try:
        print(
            f"\nProcessing {company_name} ({ticker})"
        )
        print(f"Using filing: {filing_path.name}")

        full_filing_text = clean_filing_html(
            filing_path
        )

        if len(full_filing_text) < 1_000:
            raise ValueError(
                "Very little text was extracted from the filing."
            )

        selected_filing_text = select_relevant_filing_text(
            full_filing_text,
            max_characters=MAX_FILING_CHARACTERS
        )

        filing_date = extract_filing_date(
            filing_path
        )

        prompt = create_question_generation_prompt(
            company_name=company_name,
            ticker=ticker,
            sector=sector,
            sub_industry=sub_industry,
            filing_date=filing_date,
            filing_text=selected_filing_text
        )

        questions = generate_questions(prompt)

        saved_path = save_questions(
            company_name=company_name,
            questions=questions,
            output_directory=QUESTIONS_DIR
        )

        print(
            f"Saved {len(questions)} questions to "
            f"{saved_path}"
        )

        processing_results.append(
            {
                "Ticker": ticker,
                "Company": company_name,
                "Status": "Completed",
                "Filing": filing_path.name,
                "Output": str(saved_path),
                "Error": ""
            }
        )

    except Exception as error:
        print(
            f"Failed to process {company_name} ({ticker}): "
            f"{error}"
        )

        processing_results.append(
            {
                "Ticker": ticker,
                "Company": company_name,
                "Status": "Failed",
                "Filing": filing_path.name,
                "Output": "",
                "Error": str(error)
            }
        )

    time.sleep(DELAY_BETWEEN_REQUESTS)

Generating company questions:   0%|          | 0/100 [00:00<?, ?it/s]

Skipping GOOGL - output already exists: Alphabet Inc. (Class A).txt
Skipping T - output already exists: AT&T.txt
Skipping EA - output already exists: Electronic Arts.txt
Skipping META - output already exists: Meta Platforms.txt
Skipping NFLX - output already exists: Netflix.txt
Skipping OMC - output already exists: Omnicom Group.txt
Skipping TTWO - output already exists: Take-Two Interactive.txt
Skipping DIS - output already exists: Walt Disney Company (The).txt
Skipping WBD - output already exists: Warner Bros. Discovery.txt
Skipping ABNB - output already exists: Airbnb.txt
Skipping AMZN - output already exists: Amazon.txt
Skipping BKNG - output already exists: Booking Holdings.txt
Skipping DASH - output already exists: DoorDash.txt
Skipping NKE - output already exists: Nike, Inc.txt
Skipping RL - output already exists: Ralph Lauren Corporation.txt
Skipping SBUX - output already exists: Starbucks.txt
Skipping TPR - output already exists: Tapestry, Inc.txt
Skipping TSLA - output alread

/tmp/ipykernel_260/2470591643.py:8: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(raw_content, "lxml")


Saved 30 questions to questions/Vulcan Materials Company.txt

Processing CBRE Group (CBRE)
Using filing: CBRE_10-K_2026-02-12.html
Saved 30 questions to questions/CBRE Group.txt

Processing CoStar Group (CSGP)
Using filing: CSGP_10-K_2026-02-26.html
Saved 30 questions to questions/CoStar Group.txt

Processing Iron Mountain (IRM)
Using filing: IRM_10-K_2026-02-12.html
Saved 30 questions to questions/Iron Mountain.txt

Processing Kimco Realty (KIM)
Using filing: KIM_10-K_2026-02-20.html
Attempt 1/3 failed: Expecting value: line 22 column 1 (char 2332)
Retrying in 5 seconds...
Attempt 2/3 failed: Expecting value: line 20 column 1 (char 2145)
Retrying in 10 seconds...
Saved 30 questions to questions/Kimco Realty.txt

Processing Mid-America Apartment Communities (MAA)
Using filing: MAA_10-K_2026-02-06.html
Saved 30 questions to questions/Mid-America Apartment Communities.txt

Processing Ventas (VTR)
Using filing: VTR_10-K_2026-02-06.html
Saved 30 questions to questions/Ventas.txt

Processin

In [21]:
results_df = pd.DataFrame(processing_results)

report_path = QUESTIONS_DIR / "generation_report.csv"

results_df.to_csv(
    report_path,
    index=False
)

print("\nProcessing summary:")
print(results_df["Status"].value_counts(dropna=False))

print(f"\nReport saved to: {report_path}")

display(results_df)


Processing summary:
Status
Skipped - output exists    82
Completed                  18
Name: count, dtype: int64

Report saved to: questions/generation_report.csv


,Ticker,Company,Status,Filing,Output,Error
0,GOOGL,Alphabet Inc. (Class A),Skipped - output exists,,questions/Alphabet Inc. (Class A).txt,
1,T,AT&T,Skipped - output exists,,questions/AT&T.txt,
2,EA,Electronic Arts,Skipped - output exists,,questions/Electronic Arts.txt,
3,META,Meta Platforms,Skipped - output exists,,questions/Meta Platforms.txt,
4,NFLX,Netflix,Skipped - output exists,,questions/Netflix.txt,
...,...,...,...,...,...,...
95,CEG,Constellation Energy,Completed,CEG_10-K_2026-02-24.html,questions/Constellation Energy.txt,
96,D,Dominion Energy,Completed,D_10-K_2026-02-23.html,questions/Dominion Energy.txt,
97,VST,Vistra Corp.,Completed,VST_10-K_2026-02-27.html,questions/Vistra Corp.txt,
98,WEC,WEC Energy Group,Completed,WEC_10-K_2026-02-20.html,questions/WEC Energy Group.txt,


In [22]:
failed_companies = results_df[
    results_df["Status"] == "Failed"
]

display(failed_companies)

,Ticker,Company,Status,Filing,Output,Error


In [23]:
import shutil

shutil.make_archive(
    "questions",
    "zip",
    "questions"
)

print("Created questions.zip")

Created questions.zip
